## Notebook 3: Set up a schema agnostic evaluator and golden dataset for quantitative evaluation

In this notebook, we will gradually build a `schema-agnostic evaluator` that measures the extraction quality of LLMs and IDP systems, as well as the corresponding `golden dataset` that contains a curated solution for the sample orders.

The challenge: A simple string comparison fails due to `nested JSON structures, swapped list elements, and typos`. We therefore use `Jaccard similarity on path-value tuples` combined with `fuzzy matching` and `bipartite matching`.

### Notebook Workflow
1. **Curating the `Golden Dataset` for Automated Evaluation** Construct a ground-truth JSON dataset to enable automated quantitative benchmarking.<br><br>
2. **Evaluator setup** Step by step configuration of the evaluator
- 2.1. **Schema agnostic evaluation** Extract the schema from the Golden Dataset to configurate the evaluator correctly
- 2.2. **Flattening Data Structures & None Filtering**  Preparing the data for automated evaluation
- 2.3. **Value Comparison** Configurate the best comparison options (`Exact Match` vs. `Fuzzy Match`)
- 2.4. **Comparing Tuple Sets & Calculating the `Jaccard Score`** 
- 2.5. **The Ordering Problem with Lists (`Optimal Assignment`)**
- 2.6. **Multi-Level Scoring & Reporting** Putting it all together in a single class






### **1. Curating the Golden Dataset for Automated Evaluation**

In Notebook 4, we will benchmark smaller, locally hosted models (e.g., Llama 3.1, Qwen 2.5) against our cloud extraction baseline. To perform **quantitative evaluation** automatically, we need a **Golden Dataset** (verified ground-truth JSON for our 10 test orders).

### Exercise: Generate Ground-Truth File
1. Use your refined Claude extraction results as a starting point.
2. Ask a high-capability web chatbot to look for ambiguities in the extraction result, that could be interpreted correctly in another way.
2. Review each JSON output manually against the raw transcript to verify 100% accuracy.
3. Save the verified dictionary list to `data/extraction_results/golden_dataset.json`.

In [ ]:
import json
from pathlib import Path

# Verify extracted outputs and save as ground truth
# golden_data = [ ... list of verified Pydantic dicts ... ]
golden_data = [
Paste your verified json here
]

path = Path.cwd().parent / "data" / "extraction_results" / "golden_dataset.json"
with open(path, "w", encoding="utf-8") as f:
    json.dump(golden_data, f, indent=2)
    

print(f"Golden dataset saved to: {path}")

## **2. Evaluator Setup**

### **2.1 Schema agnostic evaluation**

This step ensures that the evaluator works with different extraction schemes. <br>
Before evaluation begins, the script analyzes the Ground Truth dataset to determine the structure of the data:

1. **Identifies the Item Key (`item_list_key`):** Scans samples to find the top-level array key containing repeated target entities (e.g., `"pizzas"` or `"line_items"`).
2. **Classifies Attributes:**
* **`nested_list_fields`:** Identifies complex attributes containing dictionaries or lists of dictionaries (e.g., `topping_modifiers`).
* **`scalar_fields`:** Identifies simple attributes, including primitives (`str`, `int`, `float`, `bool`) and simple lists of primitives (`list[str]`, e.g., `special_instructions`).


3. **Set Mutual Exclusion:** Explicitly subtracts `nested_list_fields` from `scalar_fields` (`scalar_fields = scalar_fields - nested_list_fields`) so nested attributes never leak into scalar binary columns.

In [ ]:
from typing import  List, Tuple

def _infer_schema_fields(gt_dataset: List[dict]) -> Tuple[List[str], List[str], str]:
        scalar_fields = set()
        nested_list_fields = set()
        item_list_key = None

        # 1. First find item_list_key across all samples
        for order in gt_dataset:
            order_dict = order.model_dump() if hasattr(order, "model_dump") else order
            if isinstance(order_dict, dict) and item_list_key is None:
                for k, v in order_dict.items():
                    if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                        item_list_key = k
                        break

        # 2. Extract schema fields from items
        for order in gt_dataset:
            order_dict = order.model_dump() if hasattr(order, "model_dump") else order
            if not isinstance(order_dict, dict):
                continue

            items = order_dict.get(item_list_key, []) if item_list_key else (
                order_dict if isinstance(order_dict, list) else [order_dict]
            )

            for item in items:
                if isinstance(item, dict):
                    for key, val in item.items():
                        if isinstance(val, dict) or (isinstance(val, list) and any(isinstance(elem, dict) for elem in val)):
                            nested_list_fields.add(key)
                        else:
                            scalar_fields.add(key)

        scalar_fields = scalar_fields - nested_list_fields
        return sorted(list(scalar_fields)), sorted(list(nested_list_fields)), item_list_key

In [ ]:
#to test this 
from pathlib import Path
import json

gt_path = Path.cwd().parent / "data" / "extraction_results" / "golden_dataset.json"
with open(gt_path, "r", encoding="utf-8") as f:
        gt_dataset = json.load(f)

_infer_schema_fields(gt_dataset)

### **2.2 Flattening Data Structures & None Filtering**

Nested Pydantic objects are difficult to compare. <br>
For nested structures and overall item evaluation, we apply `Flattening` and `Filtering Empty Values`:

`Flattening (flatten_to_tuples):` Converts nested objects into key-path tuples (e.g., ("pizzas[0].topping_modifiers[0].ingredient", "pepperoni")).

`Filtering Empty Values:` Explicitly ignores None and empty strings "". Identical missing fields do not generate tuples, preventing unpopulated optional fields from artificially inflating accuracy scores. If None values were included, a model that extracted nothing at all would still receive a score of >80% for a large schema with many optional fields (score inflation). Only information that is actually present is evaluated.


In [ ]:
from typing import Any, List, Tuple


def flatten_to_tuples(data: Any, prefix: str = "") -> List[Tuple[str, Any]]:
    """
    Converts dictionaries, lists, and Pydantic models into a flat list of (path, value) tuples.
    Consistently filters out ‘None’ values and empty lists.
    """
    # Falls Pydantic-Modell übergeben wird, in Dict umwandeln
    if hasattr(data, "model_dump"):
        data = data.model_dump()
    elif hasattr(data, "dict"):
        data = data.dict()

    tuples = []

    if isinstance(data, dict):
        for key, value in data.items():
            new_key = f"{prefix}.{key}" if prefix else key
            tuples.extend(flatten_to_tuples(value, new_key))
    elif isinstance(data, list):
        for idx, item in enumerate(data):
            new_key = f"{prefix}[{idx}]"
            tuples.extend(flatten_to_tuples(item, new_key))
    else:
        # Nur Tupel hinzufügen, wenn ein echter Wert vorhanden ist (Kein None!)
        if data is not None and data != "":
            tuples.append((prefix, data))

    return tuples

In [ ]:
# sampble extraction of an order for 2 pizzas (PizzaOrder schema)

example_extraction = [
    {
        "pizzas": [
            {
                "is_split": True,
                "preset_name": None,
                "size": "large",
                "crust_type": "thin",
                "sauce_type": None,
                "sauce_level": "light",
                "first_half_preset": "Margherita",
                "second_half_preset": None,
                "topping_modifiers": [
                    {
                        "ingredient": "pepperoni",
                        "modifier_level": "normal",
                        "action": "add",
                        "target": "half_2"
                    }
                ],
                "special_instructions": ["Cut into 8 pieces",  "Allergy to seafood"]
            },
            {
                "is_split": False,
                "preset_name": None,
                "size": "large",
                "crust_type": "thick",
                "sauce_type": None,
                "sauce_level": "light",
                "first_half_preset": None,
                "second_half_preset": None,
                "topping_modifiers": [],
                "special_instructions": None
            }
        ]
    }
]

# Ausführung der Flattening-Funktion aus Zelle 1
flat_result = flatten_to_tuples(example_extraction)

print(f"Number of extracted datapoints: {len(flat_result)}\n")
for t in flat_result:
    print(" ", t)

### **2.3 Value Comparison (Exact Match vs. Fuzzy Match)**

Not every data type can be compared in the same way:

`Booleans, numbers, and enums/literals`: Must match exactly (“small” is not “medium”).

`Free-text / Strings`: People and LLMs use synonyms or reversed word order (“extra crispy” vs. “crispy, extra”). For evaluation, this is the most problematic challenge. Here, we apply fuzzy matching at the token level (via rapidfuzz). 

`token_set_ratio` measures string similarity by tokenizing and deduplicating words, then comparing their shared intersection against any remaining differences so that word order and extra repeated terms don't penalize the score.

A score of `70% or higher` is considered a match. Discuss specific edge cases and trade-offs of this solution with your chatbot assistant, verify that the results work well enough for your sample extraction, and `adjust the threshold` if necesary. 

Other options for the string problem: 
- embeddings with cosine similarity or 
- directly a llm as a evaluator for string fields.


In [ ]:
import math
from rapidfuzz import fuzz
from typing import Any, List, Tuple

def match_values(val1: Any, val2: Any, fuzzy_threshold: float = 70.0) -> bool:

    # 1. Numeric check (handles int vs float comparison)
    if isinstance(val1, (int, float)) and isinstance(val2, (int, float)):
        return math.isclose(val1, val2, rel_tol=1e-5)

    if type(val1) != type(val2):
        return False

    if isinstance(val1, bool):
        return val1 == val2

    # 2. Single string comparison
    if isinstance(val1, str):
        if val1.strip().lower() == val2.strip().lower():
            return True
        return fuzz.token_set_ratio(val1, val2) >= fuzzy_threshold

    # 3. List of strings / primitives comparison
    if isinstance(val1, list):
        if not val1 and not val2:
            return 1.0  # Both empty lists match perfectly
        if not val1 or not val2:
            return 0.0

        # Calculate Jaccard similarity across list elements
        matched_indices = set()
        intersection = 0
        for elem1 in val1:
            for idx2, elem2 in enumerate(val2):
                if idx2 not in matched_indices and match_values(elem1, elem2):
                    matched_indices.add(idx2)
                    intersection += 1
                    break

        union = len(val1) + len(val2) - intersection
        return round(intersection / union, 4)  # Returns a float between 0.0 and 1.0

    return val1 == val2

In [ ]:
# Check out the results for this samples: 

print("Boolean Exact Test: True vs False\n======>", match_values(True, False)) 
print()
print("Enum Exact Test: small vs medium\n======>", match_values("small", "medium")) 
print()
print("Free-Text Fuzzy Test:\nChef's special recommendation vs recommendation of the chef\n======>", match_values("Chef's special recommendation", "recommendation of the chef"))
print()
print("Free-Text Fuzzy Test:\nbox not greasy vs non-greasy box\n======>", match_values("box not greasy", "non-greasy box"))
print()
print("Free-Text Fuzzy Test:\nafter baking add  basil vs add fresh tomato after baking.\n======>", match_values("after baking Add  basil", "add fresh tomato after baking."))  
print()
print("Here comes the typical problem with fuzzy matching:")                       
print("Free-Text Fuzzy Test:\nadd onions vs no onions\n======>", match_values("add onions", "no onions"))


### 2.4: **Comparing Tuple Sets & Calculating the Jaccard Score**

To compare the total collection of tuples from two JSON objects, we use the Jaccard: <br><br>$$\text{Jaccard} = \frac{\text{intersection  } \vert{}A \cap B\vert{}}{\text{union } \vert{}A \cup B\vert{}}$$ 
<br>
**Intersection**  ($\vert{}A \cap B\vert{}$): Number of tuples where both the path AND the value match. <br>
**Union** ($\vert{}A \cup B\vert{}$): $\text{number of tuples in the Groundtruth} + \text{number of tuples extracted} - \text{intersection}$.  <br><br>
This penalizes omissions (`false negatives`) and hallucinations (`false positives`) in the denominator.





In [ ]:
from typing import Dict
import re
def compute_jaccard(gt_tuples: List[Tuple[str, Any]], ext_tuples: List[Tuple[str, Any]]) -> Dict[str, Any]:
        matched_ext_indices = set()
        intersection = 0

        for gt_k, gt_v in gt_tuples:
            gt_path_norm = re.sub(r'\[\d+\]', '[]', gt_k)
            for idx, (ext_k, ext_v) in enumerate(ext_tuples):
                if idx in matched_ext_indices:
                    continue
                ext_path_norm = re.sub(r'\[\d+\]', '[]', ext_k)
                if gt_path_norm == ext_path_norm and match_values(gt_v, ext_v):
                    intersection += 1
                    matched_ext_indices.add(idx)
                    break

        gt_count = len(gt_tuples)
        ext_count = len(ext_tuples)
        union = gt_count + ext_count - intersection
        score = (intersection / union) if union > 0 else 1.0

        return {
            "intersection": intersection,
            "gt_count": gt_count,
            "ext_count": ext_count,
            "union": union,
            "jaccard_score": round(score, 4)
        }

In [ ]:
# Ground Truth vs. Modell-Extraktion with a Hallucination
gt = [("preset_name", "Margherita"), ("size", "large"), ("special_instruction", ["no garlic", "cut in 8 pieces"])]
ext = [("preset_name", "Margherita"), ("size", "large"), ("crust_type", "stuffed"), ("special_instruction", ["cut in 8 pieces"])] 

gt = [("preset_name", "Margherita"), ("special_instruction", ["no garlic", "cut in 8 pieces"])]
ext = [("preset_name", "Margherita"), ("special_instruction", ["add onions"])] 

res = compute_jaccard(gt, ext)
print(f"Intersection: {res['intersection']}, Union: {res['union']}")
print(f"Jaccard-Score: {res['jaccard_score'] * 100}%") # Expected: 2/3 = 66.67%

### 2.5. The Ordering Problem with Lists (`Optimal Assignment`)


LLMs often extract list items in a different order than the Ground Truth, or extract too few/many items.

1. **Padding for Size Mismatches:** If Ground Truth has 3 items and Extracted has 1 item, the shorter list is padded with empty dictionaries (`{}`) up to the maximum length. This ensures omitted or hallucinated extra items are retained and scored as `0.0`.
2. **Cost Matrix Construction:** Computes pairwise structural distance between every Ground Truth item and Extracted item as $\text{Cost} = 1.0 - \text{Jaccard Score}$.
3. **Hungarian Algorithm:** Executes `scipy.optimize.linear_sum_assignment(cost_matrix)` to find the globally optimal, minimum-cost pairing between Ground Truth items and Extracted items.

In [ ]:
from scipy.optimize import linear_sum_assignment

def align_list_items(gt_list: List[Any], ext_list: List[Any]) -> List[Tuple[Any, Any]]:
    """
    Optimally matches elements from two lists,
    regardless of their order.
    """
    if not gt_list or not ext_list:
        return []

    # 1. Create a cost matrix (cost = 1.0 - Jaccard score)
    cost_matrix = []
    for gt_item in gt_list:
        row = []
        gt_flat = flatten_to_tuples(gt_item)
        for ext_item in ext_list:
            ext_flat = flatten_to_tuples(ext_item)
            score = compute_jaccard_for_tuples(gt_flat, ext_flat)["jaccard_score"]
            row.append(1.0 - score)
        cost_matrix.append(row)

    # 2. Perform Bipartite Matching
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    aligned_pairs = []
    for r, c in zip(row_ind, col_ind):
        aligned_pairs.append((gt_list[r], ext_list[c]))

    return aligned_pairs

In [ ]:
pizza_a = {"preset_name": "Margherita", "size": "small"}
pizza_b = {"preset_name": "Salami", "size": "large"}

gt_pizzas = [pizza_a, pizza_b]
ext_pizzas = [pizza_b, pizza_a] # Vertauschte Reihenfolge!

aligned = align_list_items(gt_pizzas, ext_pizzas)
print(aligned)
print("Matching pairs despite the reversed order:")
for gt_p, ext_p in aligned:
    print(f" Ground Truth: {gt_p['preset_name']} <--> Extracted: {ext_p['preset_name']}")

### 2.6 **Multi-Level Scoring & Reporting** Putting it all together in a single class


The evaluator processes the dataset and appends three distinct reporting levels to the output DataFrame:

#### Level 1: `level == "item"`

For each aligned item pair:

* **Scalar Columns:** Records match score for simple attributes (`1.0`/`0.0` for primitives, or a float $0.0$–$1.0$ for string lists).
* **Nested Sub-Score Columns (`<field>_jaccard`):** Calculates and writes an independent sub-Jaccard score for complex sub-objects (e.g., `topping_modifiers_jaccard`).
* **Overall Item `jaccard_score`:** Computes the complete Jaccard score across all valid attributes of that item combined.

#### Level 2: `level == "sample_summary"`

Evaluates the entire order/sample sample as a whole (including top-level metadata outside the item list):

* Records order-level Jaccard score in `jaccard_score`.
* Logs individual sample extraction time in `duration_sec`.
* Sets all individual field columns to `None`.

#### Level 3: `level == "dataset_summary"`

Provides global benchmark performance across the whole dataset:

* Computes **Micro-Averaged Global Jaccard Score** by summing raw intersections and unions across all samples:

$$\text{Global Jaccard} = \frac{\sum \text{Intersection}_{\text{all}}}{\sum \text{Union}_{\text{all}}}$$

* Sums valid execution times into `duration_sec`.
* Micro-averaging ensures large, complex samples carry proportionally higher weight than small, simple samples.

---

### Final Output Structure

| Column Name | Level: `item` | Level: `sample_summary` | Level: `dataset_summary` |
| --- | --- | --- | --- |
| **`dataset_name`** | File / Dataset Name | File / Dataset Name | File / Dataset Name |
| **`sample_id`** | `Sample_1`, `Sample_2` | `Sample_1`, `Sample_2` | `DATASET` |
| **`level`** | `"item"` | `"sample_summary"` | `"dataset_summary"` |
| **`item_id`** | `pizza_1`, `pizza_2` | `"-"` | `"-"` |
| **`duration_sec`** | `None` | Individual time (sec) | Total run time (sum) |
| **`<scalar_field>`** | `1.0`, `0.0`, or Float score | `None` | `None` |
| **`<nested_field>_jaccard`** | Sub-Jaccard score ($0.0$–$1.0$) | `None` | `None` |
| **`jaccard_score`** | Overall Item Jaccard Score | Sample Jaccard Score | **Global Dataset Jaccard Score** |

In [ ]:
import os
import json
import pandas as pd
from pathlib import Path
import math
import re
from typing import Any, Dict, List, Tuple, Union
from rapidfuzz import fuzz
from scipy.optimize import linear_sum_assignment

class IDPEvaluator:
    def __init__(self, fuzzy_threshold: float = 70.0):
        """
        :param fuzzy_threshold: Threshold for token set ratio in strings (0-100).
        """
        self.fuzzy_threshold = fuzzy_threshold

    def _infer_schema_fields(self, gt_dataset: List[dict]) -> Tuple[List[str], List[str], str]:
        scalar_fields = set()
        nested_list_fields = set()
        item_list_key = None

        # 1. First find item_list_key across all samples
        for order in gt_dataset:
            order_dict = order.model_dump() if hasattr(order, "model_dump") else order
            if isinstance(order_dict, dict) and item_list_key is None:
                for k, v in order_dict.items():
                    if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict):
                        item_list_key = k
                        break

        # 2. Extract schema fields from items
        for order in gt_dataset:
            order_dict = order.model_dump() if hasattr(order, "model_dump") else order
            if not isinstance(order_dict, dict):
                continue

            items = order_dict.get(item_list_key, []) if item_list_key else (
                order_dict if isinstance(order_dict, list) else [order_dict]
            )

            for item in items:
                if isinstance(item, dict):
                    for key, val in item.items():
                        if isinstance(val, dict) or (isinstance(val, list) and any(isinstance(elem, dict) for elem in val)):
                            nested_list_fields.add(key)
                        else:
                            scalar_fields.add(key)

        scalar_fields = scalar_fields - nested_list_fields
        return sorted(list(scalar_fields)), sorted(list(nested_list_fields)), item_list_key

    def match_values(self, val1: Any, val2: Any) -> Union[bool, float]:
        # 1. Numeric check
        if isinstance(val1, (int, float)) and isinstance(val2, (int, float)):
            return math.isclose(val1, val2, rel_tol=1e-5)

        if type(val1) != type(val2):
            return False

        if isinstance(val1, bool):
            return val1 == val2

        # 2. String comparison
        if isinstance(val1, str):
            if val1.strip().lower() == val2.strip().lower():
                return True
            return fuzz.token_set_ratio(val1, val2) >= self.fuzzy_threshold

        # 3. List comparison (primitive string/number lists)
        if isinstance(val1, list):
            if not val1 and not val2:
                return 1.0
            if not val1 or not val2:
                return 0.0

            matched_indices = set()
            intersection = 0
            for elem1 in val1:
                for idx2, elem2 in enumerate(val2):
                    if idx2 not in matched_indices and self.match_values(elem1, elem2):
                        matched_indices.add(idx2)
                        intersection += 1
                        break

            union = len(val1) + len(val2) - intersection
            return round(intersection / union, 4)

        return val1 == val2

    def flatten_to_tuples(self, data: Any, prefix: str = "") -> List[Tuple[str, Any]]:
        if hasattr(data, "model_dump"):
            data = data.model_dump()
        elif hasattr(data, "dict"):
            data = data.dict()

        tuples = []
        if isinstance(data, dict):
            for k, v in data.items():
                new_k = f"{prefix}.{k}" if prefix else k
                tuples.extend(self.flatten_to_tuples(v, new_k))
        elif isinstance(data, list):
            for idx, item in enumerate(data):
                new_k = f"{prefix}[{idx}]"
                tuples.extend(self.flatten_to_tuples(item, new_k))
        else:
            if data is not None and data != "":
                tuples.append((prefix, data))
        return tuples

    def compute_jaccard(self, gt_tuples: List[Tuple[str, Any]], ext_tuples: List[Tuple[str, Any]]) -> Dict[str, Any]:
        matched_ext_indices = set()
        intersection = 0

        for gt_k, gt_v in gt_tuples:
            gt_path_norm = re.sub(r'\[\d+\]', '[]', gt_k)
            for idx, (ext_k, ext_v) in enumerate(ext_tuples):
                if idx in matched_ext_indices:
                    continue
                ext_path_norm = re.sub(r'\[\d+\]', '[]', ext_k)
                if gt_path_norm == ext_path_norm and self.match_values(gt_v, ext_v):
                    intersection += 1
                    matched_ext_indices.add(idx)
                    break

        gt_count = len(gt_tuples)
        ext_count = len(ext_tuples)
        union = gt_count + ext_count - intersection
        score = (intersection / union) if union > 0 else 1.0
        

        return {
            "intersection": intersection,
            "gt_count": gt_count,
            "ext_count": ext_count,
            "union": union,
            "jaccard_score": round(score, 4)
        }

    def align_lists(self, gt_list: List[Any], ext_list: List[Any]) -> List[Tuple[Any, Any]]:
        if not gt_list and not ext_list:
            return []

        # Pad shorter list with empty dicts to account for missing/extra items
        max_len = max(len(gt_list), len(ext_list))
        gt_padded = gt_list + [{}] * (max_len - len(gt_list))
        ext_padded = ext_list + [{}] * (max_len - len(ext_list))

        cost_matrix = []
        for gt_item in gt_padded:
            row = []
            gt_flat = self.flatten_to_tuples(gt_item)
            for ext_item in ext_padded:
                ext_flat = self.flatten_to_tuples(ext_item)
                score = self.compute_jaccard(gt_flat, ext_flat)["jaccard_score"]
                row.append(1.0 - score)
            cost_matrix.append(row)

        row_ind, col_ind = linear_sum_assignment(cost_matrix)
        return [(gt_padded[r], ext_padded[c]) for r, c in zip(row_ind, col_ind)]

    def evaluate_order(self, gt_order: dict, ext_order: dict) -> dict:
        gt_flat = self.flatten_to_tuples(gt_order)
        ext_flat = self.flatten_to_tuples(ext_order)

        res = self.compute_jaccard(gt_flat, ext_flat)
        print("Intersection: ", res ["intersection"])
        print("Union: ", res ["union"])
        print("Intersection/Union: ", res ["intersection"]/ res ["union"])
        return self.compute_jaccard(gt_flat, ext_flat)

    def evaluate_and_export(
        self, 
        gt_dataset: List[dict], 
        ext_dataset: List[dict], 
        dataset_name: str,
        output_path: str,
        sample_label: str = "Sample",
        duration_list: List[float] = None
    ) -> pd.DataFrame:
        scalar_fields, nested_list_fields, item_list_key = self._infer_schema_fields(gt_dataset)

        rows = []
        total_intersection = 0
        total_gt = 0
        total_ext = 0

        for sample_idx, (gt_sample, ext_sample) in enumerate(zip(gt_dataset, ext_dataset), start=1):
            sample_id = f"{sample_label}_{sample_idx}"
            
            current_duration = (
                duration_list[sample_idx - 1] 
                if duration_list and (sample_idx - 1) < len(duration_list) 
                else None
            )

            gt_dict = gt_sample.model_dump() if hasattr(gt_sample, "model_dump") else gt_sample
            ext_dict = ext_sample.model_dump() if hasattr(ext_sample, "model_dump") else ext_sample

            if item_list_key:
                gt_items = gt_dict.get(item_list_key, [])
                ext_items = ext_dict.get(item_list_key, [])
            else:
                gt_items = [gt_dict] if isinstance(gt_dict, dict) else gt_dict
                ext_items = [ext_dict] if isinstance(ext_dict, dict) else ext_dict

            aligned_items = self.align_lists(gt_items, ext_items)

            # 1. ITEM-EBENE
            for item_idx, (gt_item, ext_item) in enumerate(aligned_items, start=1):
                item_label = f"{item_list_key[:-1] if item_list_key and item_list_key.endswith('s') else 'item'}_{item_idx}"
                row = {
                    "dataset_name": dataset_name,
                    "sample_id": sample_id,
                    "level": "item",
                    "item_id": item_label,
                    "duration_sec": None  
                }

                for field in scalar_fields:
                    gt_v = gt_item.get(field)
                    ext_v = ext_item.get(field)
                    if gt_v is None and ext_v is None:
                        row[field] = 1.0
                    elif gt_v is not None and ext_v is not None:
                        match_res = self.match_values(gt_v, ext_v)
                        row[field] = float(match_res) if isinstance(match_res, (bool, int, float)) else (1.0 if match_res else 0.0)
                    else:
                        row[field] = 0.0

                for list_field in nested_list_fields:
                    gt_nested = self.flatten_to_tuples(gt_item.get(list_field, []))
                    ext_nested = self.flatten_to_tuples(ext_item.get(list_field, []))
                    row[f"{list_field}_jaccard"] = self.compute_jaccard(gt_nested, ext_nested)["jaccard_score"]

                item_gt_flat = self.flatten_to_tuples(gt_item)
                item_ext_flat = self.flatten_to_tuples(ext_item)
                row["jaccard_score"] = self.compute_jaccard(item_gt_flat, item_ext_flat)["jaccard_score"]
                rows.append(row)

            # 2. SAMPLE-EBENE
            sample_metrics = self.evaluate_order(gt_dict, ext_dict)
            total_intersection += sample_metrics["intersection"]
            total_gt += sample_metrics["gt_count"]
            total_ext += sample_metrics["ext_count"]

            sample_row = {
                "dataset_name": dataset_name,
                "sample_id": sample_id,
                "level": "sample_summary",
                "item_id": "-",
                "duration_sec": round(current_duration, 4) if current_duration is not None else None,
                "jaccard_score": sample_metrics["jaccard_score"]
            }
            for field in scalar_fields + [f"{lf}_jaccard" for lf in nested_list_fields]:
                sample_row[field] = None
            rows.append(sample_row)

        # 3. DATASET-EBENE
        total_union = total_gt + total_ext - total_intersection
        global_jaccard = (total_intersection / total_union) if total_union > 0 else 1.0
        total_duration = sum(d for d in duration_list if d is not None) if duration_list else None

        ds_row = {
            "dataset_name": dataset_name,
            "sample_id": "DATASET",
            "level": "dataset_summary",
            "item_id": "-",
            "duration_sec": round(total_duration, 4) if total_duration is not None else None,
            "jaccard_score": round(global_jaccard, 4)
        }
        for field in scalar_fields + [f"{lf}_jaccard" for lf in nested_list_fields]:
            ds_row[field] = None
        rows.append(ds_row)     
   
        # Convert to DataFrame & save CSV
        if os.path.exists(output_path):
            df_old = pd.read_csv(output_path, encoding="utf-8-sig")
            df_new = pd.DataFrame(rows)
            df = pd.concat([df_old, df_new], ignore_index=True)
        else:
            df = pd.DataFrame(rows)

        #df.to_csv(output_path, index=False, encoding="utf-8-sig")
        
        #print(f"✅ Evaluation complete. Results saved in '{output_path}'.")
        print(f"📊 Global Dataset Jaccard Score: {round(global_jaccard * 100, 2)}%")
        
        return df

In [ ]:
#test with a invoice data extractor

gt_dataset = [
    {
        "line_items": [
            {
                "description": "Wireless Ergonomic Keyboard",
                "unit_price": 49.99,
                "tags": ["electronics", "taxable", "Madrid"], #Madrid is missing in the extraction
                "tax_breakdown": [
                    {"tax_type": "VAT", "rate": 19.0}
                ]
            }
        ]
    }
]

ext_dataset = [
    {
        "line_items": [
            {
                "description": "wireless ergonomic keyboard",
                "unit_price": 49.99,
                "tags": ["taxable", "electronics", "Spain"], # Spain is a haluzination
                "tax_breakdown": [
                    {"tax_type": "VAT", "rate": 19.00}
                ]
            }
        ]
    }
]

evaluator = IDPEvaluator(fuzzy_threshold=70.0)

df = evaluator.evaluate_and_export(
    gt_dataset=gt_dataset,
    ext_dataset=ext_dataset,
    dataset_name="invoice_test_run",
    output_path="invoice_eval_results.csv",
    duration_list=[0.85]
)

print(df.iloc[0])

# Union:8 Intersection: 6 = 0,75